# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
from src.utils import config, io

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# 2 - Preprocessing

In [4]:
X = io.load_csv(config.PROCESSED_DATA_DIR / 'X.csv', index_col=0)
y = io.load_csv(config.PROCESSED_DATA_DIR / 'y.csv', index_col=0)

In [5]:
split_cfg = io.load_json(config.PROCESSED_DATA_DIR / 'splits/temporal_v1.json')
split_cfg

{'description': 'Forecasting split for future country risk prediction',
 'train_years': [1999, 2015],
 'val_years': [2016, 2018],
 'test_years': [2019, 2024]}

In [6]:
def get_subset_data(data, bounds):
    return data[(data['YEAR'] >= bounds[0]) & (data['YEAR'] <= bounds[1])]

In [7]:
X_train = get_subset_data(X, split_cfg['train_years'])
y_train = y.loc[X_train.index]
X_val = get_subset_data(X, split_cfg['val_years'])
y_val = y.loc[X_val.index]
X_test = get_subset_data(X, split_cfg['test_years'])
y_test = y.loc[X_test.index]

# 3 - Train Model

In [9]:
from src.preprocessing import preprocess_pipeline
from src.models import model_pipeline, evaluate

In [14]:
%pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
preprocessor = preprocess_pipeline.build_preprocessor(X)

In [11]:
import mlflow

mlflow.set_tracking_uri(config.PROJECT_ROOT / 'models/mlruns')
mlflow.set_experiment('Country Risk Prediction')


/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


<Experiment: artifact_location='file:///Users/hippolytegrandet/Desktop/Dev/country_risk_rating/models/mlruns/991689756472581023', creation_time=1770566631957, experiment_id='991689756472581023', last_update_time=1770566631957, lifecycle_stage='active', name='Country Risk Prediction', tags={}>

## 3.1 - Baseline, Logistic Regression Model

In [12]:
model_name = 'logistic_regression'

model_params = {
    'C': 1.0,
    'max_iter': 1000,
    'class_weight': 'balanced'
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

In [ ]:
with mlflow.start_run(run_name='baseline_lr_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    val_metrics = evaluate.evaluate_model(model, X_val, y_val, prefix='val_')
    test_metrics = evaluate.evaluate_model(model, X_test, y_test, prefix='test_')

    mlflow.log_metrics({**val_metrics, **test_metrics})

    # Log model
    # mlflow.sklearn.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

2026/02/08 17:19:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validat

Baseline Logistic Regression Results
val_accuracy: 0.6839
val_precision: 0.6272
val_recall: 0.6372
val_f1: 0.6255

Classification Report
              precision    recall  f1-score   support

           1       0.93      0.69      0.79       162
           2       0.67      0.58      0.62        24
           3       0.55      0.72      0.62        46
           4       0.46      0.42      0.44        31
           5       0.45      0.60      0.52        40
           6       0.56      0.65      0.60        94
           7       0.76      0.80      0.78       125

    accuracy                           0.68       522
   macro avg       0.63      0.64      0.63       522
weighted avg       0.71      0.68      0.69       522


Confusion Matrix
[[112   1   2   4   1  21  21]
 [  2  14   8   0   0   0   0]
 [  1   6  33   6   0   0   0]
 [  1   0  13  13   4   0   0]
 [  0   0   4   1  24  10   1]
 [  0   0   0   1  23  61   9]
 [  4   0   0   3   1  17 100]]


2026/02/08 17:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Baseline Logistic Regression Results
test_accuracy: 0.6090
test_precision: 0.5080
test_recall: 0.4956
test_f1: 0.4931

Classification Report
              precision    recall  f1-score   support

           1       0.78      0.70      0.74       270
           2       0.50      0.33      0.40        33
           3       0.53      0.64      0.58        73
           4       0.12      0.07      0.08        46
           5       0.44      0.36      0.39        84
           6       0.44      0.67      0.53       139
           7       0.74      0.70      0.72       217

    accuracy                           0.61       862
   macro avg       0.51      0.50      0.49       862
weighted avg       0.62      0.61      0.61       862


Confusion Matrix
[[189   1   2   8   0  33  37]
 [ 11  11  11   0   0   0   0]
 [ 13   6  47   2   2   3   0]
 [  6   1  15   3  15   6   0]
 [  4   2   7   8  30  32   1]
 [  6   1   3   3  17  93  16]
 [ 12   0   3   1   4  45 152]]


/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in

MLflow run_id: 19e7ce582e774588a933156a2aadfff2


In [17]:
model['preprocessing']

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer', KNNImputer()),
                                                 ('scaler', StandardScaler()),
                                                 ('norm', Normalizer())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x147f9ac40>),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x147f9abe0>)])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median']
}

search_cv = RandomizedSearchCV(model, param_grid, n_iter=10)

## 3.2 XGBoost Classifier

In [ ]:
model_name = 'xgboost'

model_params = {
    'reg_alpha': 0.01,
    'colsample_bytree': 0.60,
    'eta': 0.3,
    'eval_metric': ['mlogloss'],
    'gamma': 0.00001,
    'reg_lambda': 0.0001,
    'max_depth': 6,
    'min_child_weight': 0.2,
    'num_class': 7,
    'subsample': 0.7
}

model = model_pipeline.get_model_pipeline(
    model_name,
    preprocessor,
    model_params
)

AttributeError: module 'mlflow.xgboost' has no attribute 'XGBClassifier'

In [ ]:
with mlflow.start_run(run_name='baseline_xgb_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    val_metrics = evaluate.evaluate_model(model, X_val, y_val, prefix='val_')
    test_metrics = evaluate.evaluate_model(model, X_test, y_test, prefix='test_')

    mlflow.log_metrics({**val_metrics, **test_metrics})

    # Log model
    # mlflow.xgboost.log_model(
    #     model,
    #     artifact_path='model',
    #     registered_model_name=None,
    #     input_example=X_test.loc[[X_test.index[0]]]
    # )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

2026/02/08 17:19:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validat

Baseline Logistic Regression Results
val_accuracy: 0.6839
val_precision: 0.6272
val_recall: 0.6372
val_f1: 0.6255

Classification Report
              precision    recall  f1-score   support

           1       0.93      0.69      0.79       162
           2       0.67      0.58      0.62        24
           3       0.55      0.72      0.62        46
           4       0.46      0.42      0.44        31
           5       0.45      0.60      0.52        40
           6       0.56      0.65      0.60        94
           7       0.76      0.80      0.78       125

    accuracy                           0.68       522
   macro avg       0.63      0.64      0.63       522
weighted avg       0.71      0.68      0.69       522


Confusion Matrix
[[112   1   2   4   1  21  21]
 [  2  14   8   0   0   0   0]
 [  1   6  33   6   0   0   0]
 [  1   0  13  13   4   0   0]
 [  0   0   4   1  24  10   1]
 [  0   0   0   1  23  61   9]
 [  4   0   0   3   1  17 100]]


2026/02/08 17:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Baseline Logistic Regression Results
test_accuracy: 0.6090
test_precision: 0.5080
test_recall: 0.4956
test_f1: 0.4931

Classification Report
              precision    recall  f1-score   support

           1       0.78      0.70      0.74       270
           2       0.50      0.33      0.40        33
           3       0.53      0.64      0.58        73
           4       0.12      0.07      0.08        46
           5       0.44      0.36      0.39        84
           6       0.44      0.67      0.53       139
           7       0.74      0.70      0.72       217

    accuracy                           0.61       862
   macro avg       0.51      0.50      0.49       862
weighted avg       0.62      0.61      0.61       862


Confusion Matrix
[[189   1   2   8   0  33  37]
 [ 11  11  11   0   0   0   0]
 [ 13   6  47   2   2   3   0]
 [  6   1  15   3  15   6   0]
 [  4   2   7   8  30  32   1]
 [  6   1   3   3  17  93  16]
 [ 12   0   3   1   4  45 152]]


/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in

MLflow run_id: 19e7ce582e774588a933156a2aadfff2


## 3.3 - PyTorch NN Model

# 4 - Save Model in Registry

In [ ]:
from src.models import registry

In [ ]:
registry.save_to_registry(
    model, 
    model_name='baseline_lr', 
    framework='sklearn', 
    test_metrics=test_metrics, 
    run_id=run_id, 
    version=1
)